In [ ]:
# C6. Data Quality Assessment

## Objective

The objective of this notebook is to evaluate the overall quality of the cleaned operational dataset before it is used for exploratory data analysis, feature engineering, statistical investigation, and machine learning.

This notebook assesses the completeness, consistency, validity, and integrity of the cleaned dataset to ensure that the recorded PEM fuel cell measurements are reliable and suitable for subsequent analysis.

Unlike the previous notebook, which focused on cleaning and standardising the dataset, this notebook evaluates the quality of the cleaned data without modifying the experimental observations.

In [ ]:
## Workflow

This notebook covers the following tasks:

1. Import required libraries
2. Configure display settings
3. Define project paths
4. Load the cleaned operational dataset
5. Assess dataset completeness
6. Assess dataset consistency
7. Assess measurement validity
8. Assess dataset integrity
9. Generate the data quality summary
10. Record key findings and initial observations

In [1]:
# ============================================================
# Import Required Libraries
# ============================================================

from pathlib import Path

import pandas as pd
import numpy as np

In [2]:
# ============================================================
# Configure Display Settings
# ============================================================

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", None)

In [3]:
# ============================================================
# Define Project Paths
# ============================================================

PROJECT_ROOT = Path.cwd().parent

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

print("Project Root:", PROJECT_ROOT)
print("Processed Data Folder:", PROCESSED_DATA_DIR)

Project Root: C:\Users\usman\Desktop\PEMFC_Dissertation
Processed Data Folder: C:\Users\usman\Desktop\PEMFC_Dissertation\data\processed


In [ ]:
## C6.1 Load the Cleaned Dataset

The cleaned operational dataset produced in Notebook 04 is loaded for data quality assessment.

This dataset has already undergone structural cleaning and data type conversion. The objective of this notebook is to evaluate its quality rather than modify its contents.

In [4]:
# ============================================================
# Load Cleaned Operational Dataset
# ============================================================

cleaned_dataset = pd.read_csv(
    PROCESSED_DATA_DIR / "operational_cleaned.csv",
    low_memory=False
)

print("Cleaned operational dataset loaded successfully.")

Cleaned operational dataset loaded successfully.


In [ ]:
## C6.1.1 Verify Successful Loading

Before assessing data quality, the cleaned dataset is verified to confirm that it has been loaded successfully and that its overall dimensions match the expected output from Notebook 04.

This verification ensures that the correct version of the dataset is being used throughout the quality assessment process.

In [5]:
# ============================================================
# Verify Successful Dataset Loading
# ============================================================

print("Dataset Shape:", cleaned_dataset.shape)

print("\nNumber of Observations:", f"{cleaned_dataset.shape[0]:,}")

print("Number of Variables:", cleaned_dataset.shape[1])

print("Operating-Hour Experiments:",
      cleaned_dataset["operating_hour"].nunique())

Dataset Shape: (3629680, 18)

Number of Observations: 3,629,680
Number of Variables: 18
Operating-Hour Experiments: 20


In [ ]:
## C6.2 Completeness Assessment

Completeness is one of the fundamental dimensions of data quality.

A complete dataset contains all expected observations and measurement values required for subsequent analysis. Missing information may reduce the reliability of statistical analysis, feature engineering, and predictive modelling.

The objective of this section is to evaluate the completeness of the cleaned operational dataset by examining:

- overall dataset completeness;
- completeness of individual variables;
- completeness of each operating-hour experiment.

This assessment provides confidence that the dataset contains sufficient information for subsequent analysis.

In [ ]:
### Overall Dataset Completeness

The overall completeness of the dataset is assessed by calculating the total number of missing values and the overall percentage of complete data.

This provides a high-level indication of the dataset's overall quality before examining individual variables and experiments.

In [7]:
# ============================================================
# Overall Dataset Completeness
# ============================================================

total_cells = cleaned_dataset.size

missing_cells = cleaned_dataset.isnull().sum().sum()

complete_cells = total_cells - missing_cells

completeness_percentage = (
    complete_cells / total_cells
) * 100

overall_completeness = pd.DataFrame({

    "Metric": [
        "Total cells",
        "Complete cells",
        "Missing cells",
        "Overall completeness (%)"
    ],

    "Value": [
        f"{total_cells:,}",
        f"{complete_cells:,}",
        f"{missing_cells:,}",
        f"{completeness_percentage:.2f}%"
    ]

})

overall_completeness

,Metric,Value
0,Total cells,"65,334,240"
1,Complete cells,"65,334,240"
2,Missing cells,0
3,Overall completeness (%),100.00%


In [ ]:
### Variable Completeness

The completeness of each variable is assessed by calculating the number of missing values and the percentage of complete observations.

This helps identify whether any individual measurement variable contains incomplete information.

In [8]:
# ============================================================
# Variable Completeness
# ============================================================

variable_completeness = pd.DataFrame({

    "Variable": cleaned_dataset.columns,

    "Missing Values": cleaned_dataset.isnull().sum().values,

    "Complete Values": cleaned_dataset.notnull().sum().values

})

variable_completeness["Completeness (%)"] = (

    variable_completeness["Complete Values"]

    /

    len(cleaned_dataset)

) * 100

variable_completeness

,Variable,Missing Values,Complete Values,Completeness (%)
0,operating_hour,0,3629680,100.0
1,time,0,3629680,100.0
2,current,0,3629680,100.0
3,voltage,0,3629680,100.0
4,power,0,3629680,100.0
5,pressure_anode_inlet,0,3629680,100.0
6,pressure_anode_outlet,0,3629680,100.0
7,pressure_cathode_inlet,0,3629680,100.0
8,pressure_cathode_outlet,0,3629680,100.0
9,temp_anode_endplate,0,3629680,100.0


In [ ]:
### Experimental Completeness

The completeness of each operating-hour experiment is assessed individually.

This verifies that no experiment contains incomplete observations that could influence later comparisons between different degradation stages.

In [9]:
# ============================================================
# Experimental Completeness
# ============================================================

experiment_completeness = (

    cleaned_dataset

    .groupby("operating_hour")

    .apply(

        lambda df: pd.Series({

            "Rows": len(df),

            "Missing Values": df.isnull().sum().sum(),

            "Completeness (%)": (

                1 -

                df.isnull().sum().sum()

                /

                df.size

            ) * 100

        })

    )

    .reset_index()

)

experiment_completeness

,operating_hour,Rows,Missing Values,Completeness (%)
0,50,179360.0,0.0,100.0
1,100,179360.0,0.0,100.0
2,150,179360.0,0.0,100.0
3,200,179360.0,0.0,100.0
4,250,179360.0,0.0,100.0
5,300,179360.0,0.0,100.0
6,350,179360.0,0.0,100.0
7,400,179360.0,0.0,100.0
8,450,179360.0,0.0,100.0
9,500,179360.0,0.0,100.0


In [ ]:
### Completeness Summary

The completeness assessment summarises the overall availability of data within the cleaned operational dataset.

This summary confirms whether sufficient information is available for exploratory analysis and predictive modelling.

In [10]:
# ============================================================
# Completeness Summary
# ============================================================

completeness_summary = pd.DataFrame({

    "Metric": [

        "Overall completeness",

        "Variables with missing values",

        "Experiments with missing values",

        "Dataset status"

    ],

    "Value": [

        f"{completeness_percentage:.2f}%",

        (variable_completeness["Missing Values"] > 0).sum(),

        (experiment_completeness["Missing Values"] > 0).sum(),

        "Complete and suitable for further analysis"

        if missing_cells == 0

        else "Contains missing values"

    ]

})

completeness_summary

,Metric,Value
0,Overall completeness,100.00%
1,Variables with missing values,0
2,Experiments with missing values,0
3,Dataset status,Complete and suitable for further analysis


In [ ]:
## C6.3 Consistency Assessment

Consistency is another fundamental dimension of data quality.

A consistent dataset maintains the same structure, formatting, data types, and identifiers throughout all observations. Consistency ensures that every operating-hour experiment can be analysed together without introducing structural bias or inconsistencies.

The objective of this section is to verify that the cleaned operational dataset remains structurally consistent after the cleaning process.

In [ ]:
### Variable Name Consistency

The variable names are reviewed to confirm that each measurement variable has been preserved correctly and that no unintended changes occurred during the cleaning process.

In [11]:
# ============================================================
# Variable Name Consistency
# ============================================================

variable_names = pd.DataFrame({
    "Variable": cleaned_dataset.columns
})

variable_names

,Variable
0,operating_hour
1,time
2,current
3,voltage
4,power
5,pressure_anode_inlet
6,pressure_anode_outlet
7,pressure_cathode_inlet
8,pressure_cathode_outlet
9,temp_anode_endplate


In [ ]:
### Data Type Consistency

The data types of all variables are verified to ensure that operational measurements remain stored as numerical variables while the experimental identifier is retained as an integer.

In [12]:
# ============================================================
# Data Type Consistency
# ============================================================

datatype_summary = pd.DataFrame({

    "Variable": cleaned_dataset.columns,

    "Data Type": cleaned_dataset.dtypes.astype(str).values

})

datatype_summary

,Variable,Data Type
0,operating_hour,int64
1,time,float64
2,current,float64
3,voltage,float64
4,power,float64
5,pressure_anode_inlet,float64
6,pressure_anode_outlet,float64
7,pressure_cathode_inlet,float64
8,pressure_cathode_outlet,float64
9,temp_anode_endplate,float64


In [ ]:
### Operating-Hour Identifier Consistency

The operating-hour identifier is examined to verify that all expected experiments remain present and that no experimental identities were altered during preprocessing.

In [13]:
# ============================================================
# Operating-Hour Identifier Consistency
# ============================================================

operating_hour_summary = (

    cleaned_dataset

    .groupby("operating_hour")

    .size()

    .reset_index(name="Observations")

)

operating_hour_summary

,operating_hour,Observations
0,50,179360
1,100,179360
2,150,179360
3,200,179360
4,250,179360
5,300,179360
6,350,179360
7,400,179360
8,450,179360
9,500,179360


In [ ]:
### Dataset Structure Consistency

The overall structure of the cleaned dataset is verified to confirm that the expected variables remain present and that the dataset retains a consistent format suitable for statistical analysis and machine learning.

In [14]:
# ============================================================
# Dataset Structure Consistency
# ============================================================

structure_summary = pd.DataFrame({

    "Metric": [

        "Variables",
        "Numerical variables",
        "Identifier variables",
        "Duplicate variable names"

    ],

    "Value": [

        cleaned_dataset.shape[1],

        len(
            cleaned_dataset.select_dtypes(include="number").columns
        ),

        1,

        cleaned_dataset.columns.duplicated().sum()

    ]

})

structure_summary

,Metric,Value
0,Variables,18
1,Numerical variables,18
2,Identifier variables,1
3,Duplicate variable names,0


In [ ]:
### Consistency Summary

The consistency assessment summarises the structural consistency of the cleaned operational dataset.

This confirms that all variables, identifiers, and data types remain consistent across the complete dataset following the cleaning process.

In [15]:
# ============================================================
# Consistency Summary
# ============================================================

consistency_summary = pd.DataFrame({

    "Metric": [

        "Variable name consistency",
        "Data type consistency",
        "Operating-hour consistency",
        "Duplicate variable names",
        "Dataset consistency status",
        "Dataset readiness"

    ],

    "Value": [

        "18 variables verified",

        "17 float64 variables and 1 int64 identifier verified",

        "20 operating-hour experiments verified",

        cleaned_dataset.columns.duplicated().sum(),

        "Structurally consistent",

        "Ready for Validity Assessment (C6.4)"

    ]

})

consistency_summary

,Metric,Value
0,Variable name consistency,18 variables verified
1,Data type consistency,17 float64 variables and 1 int64 identifier verified
2,Operating-hour consistency,20 operating-hour experiments verified
3,Duplicate variable names,0
4,Dataset consistency status,Structurally consistent
5,Dataset readiness,Ready for Validity Assessment (C6.4)


In [ ]:
## C6.4 Validity Assessment

Validity refers to whether the recorded measurements represent physically plausible operating conditions.

Unlike completeness and consistency, which evaluate the structure of the dataset, validity evaluates the measurements themselves.

The objective of this section is to determine whether the recorded voltage, current, power, pressure, temperature, and reactant flow measurements fall within realistic engineering operating ranges.

Measurements identified as unusual are not automatically considered errors. Instead, they are highlighted for further investigation during exploratory data analysis.

In [ ]:
### Measurement Range Assessment

The minimum and maximum values of every operational measurement are examined to understand the observed operating ranges within the cleaned dataset.

This assessment provides an initial indication of whether any measurements appear unusual and establishes reference ranges for subsequent exploratory analysis.

In [16]:
# ============================================================
# Measurement Range Assessment
# ============================================================

range_summary = pd.DataFrame({

    "Variable": cleaned_dataset.columns,

    "Minimum": cleaned_dataset.min().values,

    "Maximum": cleaned_dataset.max().values

})

range_summary

,Variable,Minimum,Maximum
0,operating_hour,50.000000,1000.000000
1,time,1.126000,221840.246000
2,current,-0.002500,35.541400
3,voltage,-0.058700,0.949800
4,power,-0.870000,21.780000
5,pressure_anode_inlet,49.069857,126.797522
6,pressure_anode_outlet,49.438740,126.200220
7,pressure_cathode_inlet,100.479385,127.809272
8,pressure_cathode_outlet,99.259828,111.323930
9,temp_anode_endplate,81.406158,84.978310


In [ ]:
### Fundamental Physical Constraint Assessment

The cleaned dataset is examined for measurements that are physically implausible.

This assessment focuses on identifying impossible values rather than unusual operating conditions.

Any observations identified during this assessment will be documented for further investigation rather than removed.

In [17]:
# ============================================================
# Physical Validity Assessment
# ============================================================

total_observations = len(cleaned_dataset)

validity_summary = pd.DataFrame({

    "Variable": [
        "Voltage",
        "Current",
        "Power",
        "Pressure",
        "Temperature",
        "Flow"
    ],

    "Check": [
        "Negative voltage",
        "Negative current",
        "Negative power",
        "Negative pressure",
        "Below absolute zero",
        "Negative flow"
    ],

    "Count": [
        (cleaned_dataset["voltage"] < 0).sum(),
        (cleaned_dataset["current"] < 0).sum(),
        (cleaned_dataset["power"] < 0).sum(),
        (cleaned_dataset.filter(like="pressure") < 0).sum().sum(),
        (cleaned_dataset.filter(like="temp") < -273.15).sum().sum(),
        (cleaned_dataset.filter(like="flow") < 0).sum().sum()
    ]

})

validity_summary["Percentage (%)"] = (
    validity_summary["Count"] / total_observations * 100
).round(4)

validity_summary

,Variable,Check,Count,Percentage (%)
0,Voltage,Negative voltage,1,0.0000
1,Current,Negative current,1858,0.0512
2,Power,Negative power,1,0.0000
3,Pressure,Negative pressure,0,0.0000
4,Temperature,Below absolute zero,0,0.0000
5,Flow,Negative flow,0,0.0000


In [ ]:
### Variable Range Summary

The observed measurement ranges provide an overview of the operating conditions represented within the dataset.

These ranges will later be compared with engineering expectations reported in the PEM fuel cell literature during exploratory data analysis.

In [18]:
range_summary

,Variable,Minimum,Maximum
0,operating_hour,50.000000,1000.000000
1,time,1.126000,221840.246000
2,current,-0.002500,35.541400
3,voltage,-0.058700,0.949800
4,power,-0.870000,21.780000
5,pressure_anode_inlet,49.069857,126.797522
6,pressure_anode_outlet,49.438740,126.200220
7,pressure_cathode_inlet,100.479385,127.809272
8,pressure_cathode_outlet,99.259828,111.323930
9,temp_anode_endplate,81.406158,84.978310


In [ ]:
### Validity Summary

The validity assessment summarises whether the cleaned dataset contains any obviously impossible measurements.

The objective is not to judge engineering performance at this stage but to confirm that the recorded measurements represent physically plausible operating conditions suitable for further investigation.

In [19]:
# ============================================================
# Validity Summary
# ============================================================

total_invalid = validity_summary["Count"].sum()

validity_report = pd.DataFrame({

    "Metric":[

        "Variables assessed",

        "Impossible measurements detected",

        "Overall validity status",

        "Dataset readiness"

    ],

    "Value":[

        17,

        total_invalid,

        "Physically plausible"
        if total_invalid == 0
        else "Requires investigation",

        "Ready for Dataset Integrity Assessment (C6.5)"

    ]

})

validity_report

,Metric,Value
0,Variables assessed,17
1,Impossible measurements detected,1860
2,Overall validity status,Requires investigation
3,Dataset readiness,Ready for Dataset Integrity Assessment (C6.5)


In [24]:
# ============================================================
# Experimental Operating Condition Verification
# ============================================================

operating_conditions = pd.DataFrame({

    "Variable": [

        "Anode Inlet Pressure",
        "Cathode Inlet Pressure",
        "Anode Inlet Temperature",
        "Cathode Inlet Temperature",
        "Anode Dew Point",
        "Cathode Dew Point"

    ],

    "Dataset Variable":[

        "pressure_anode_inlet",
        "pressure_cathode_inlet",
        "temp_anode_inlet",
        "temp_cathode_inlet",
        "temp_anode_dewpoint_water",
        "temp_cathode_dewpoint_water"

    ],

    "Experimental Setpoint":[

        110,
        110,
        70,
        70,
        55,
        65

    ]

})

results = []

for _, row in operating_conditions.iterrows():

    variable = row["Dataset Variable"]

    results.append({

        "Variable": row["Variable"],
        "Setpoint": row["Experimental Setpoint"],
        "Minimum": cleaned_dataset[variable].min(),
        "Maximum": cleaned_dataset[variable].max(),
        "Mean": cleaned_dataset[variable].mean(),
        "Median": cleaned_dataset[variable].median(),
        "Standard Deviation": cleaned_dataset[variable].std()

    })

operating_summary = pd.DataFrame(results)

operating_summary.round(3)

,Variable,Setpoint,Minimum,Maximum,Mean,Median,Standard Deviation
0,Anode Inlet Pressure,110,49.070,126.798,109.955,109.901,1.238
1,Cathode Inlet Pressure,110,100.479,127.809,109.819,109.800,1.490
2,Anode Inlet Temperature,70,63.106,76.734,69.983,69.937,0.437
3,Cathode Inlet Temperature,70,67.198,72.854,69.993,69.753,0.783
4,Anode Dew Point,55,52.340,56.525,54.952,54.993,0.209
5,Cathode Dew Point,65,58.791,66.026,64.955,64.988,0.237


In [ ]:
## C6.5 Dataset Integrity Assessment

Dataset integrity refers to the preservation of the original experimental information throughout the preprocessing workflow.

Following data merging and cleaning, it is essential to verify that the dataset remains complete, that all operating-hour experiments are preserved, and that no unintended structural changes have occurred.

This assessment provides the final confirmation that the cleaned dataset accurately represents the original operational experiments before exploratory data analysis begins.

In [21]:
# Calculate power from the recorded voltage and current
cleaned_dataset["calculated_power"] = (
    cleaned_dataset["voltage"] * cleaned_dataset["current"]
)

# Count negative values
negative_calculated = (cleaned_dataset["calculated_power"] < 0).sum()
negative_recorded = (cleaned_dataset["power"] < 0).sum()

print("Negative Calculated Power :", negative_calculated)
print("Negative Recorded Power   :", negative_recorded)

Negative Calculated Power : 1859
Negative Recorded Power   : 1


In [22]:
cleaned_dataset.loc[
    cleaned_dataset["current"] < 0,
    ["voltage", "current", "power", "calculated_power"]
].head(20)

,voltage,current,power,calculated_power
23595,0.9436,-0.0025,0.0,-0.002359
30672,0.9430,-0.0025,0.0,-0.002357
33054,0.9414,-0.0025,0.0,-0.002354
36570,0.9433,-0.0025,0.0,-0.002358
44853,0.9407,-0.0025,0.0,-0.002352
46033,0.9414,-0.0025,0.0,-0.002354
48393,0.9404,-0.0025,0.0,-0.002351
55463,0.9417,-0.0025,0.0,-0.002354
60175,0.9410,-0.0025,0.0,-0.002352
62526,0.9433,-0.0025,0.0,-0.002358


In [23]:
cleaned_dataset["power_difference"] = (
    cleaned_dataset["power"] -
    cleaned_dataset["calculated_power"]
)

cleaned_dataset["power_difference"].describe()

count    3.629680e+06
mean    -1.331254e-05
std      2.867890e-03
min     -6.708120e-03
25%     -2.402040e-03
50%      0.000000e+00
75%      2.405480e-03
max      6.668060e-03
Name: power_difference, dtype: float64

In [ ]:
### Dataset Dimension Verification

The dimensions of the cleaned dataset are verified to confirm that the expected number of observations and variables remain after preprocessing.

In [22]:
# ============================================================
# Dataset Dimension Verification
# ============================================================

dimension_summary = pd.DataFrame({

    "Metric": [

        "Total observations",
        "Total variables"

    ],

    "Value": [

        f"{cleaned_dataset.shape[0]:,}",
        cleaned_dataset.shape[1]

    ]

})

dimension_summary

,Metric,Value
0,Total observations,"3,629,680"
1,Total variables,18


In [ ]:
### Operating-Hour Integrity

The operating-hour identifiers are verified to ensure that every experimental condition remains present within the cleaned dataset.

In [23]:
# ============================================================
# Operating-Hour Integrity
# ============================================================

integrity_experiments = pd.DataFrame({

    "Metric":[

        "Operating-hour experiments",

        "Minimum operating hour",

        "Maximum operating hour"

    ],

    "Value":[

        cleaned_dataset["operating_hour"].nunique(),

        cleaned_dataset["operating_hour"].min(),

        cleaned_dataset["operating_hour"].max()

    ]

})

integrity_experiments

,Metric,Value
0,Operating-hour experiments,20
1,Minimum operating hour,50
2,Maximum operating hour,1000


In [ ]:
### Dataset Preservation Verification

The cleaned dataset is examined to confirm that every operating-hour experiment remains represented after preprocessing.

This verification demonstrates that the preprocessing procedures preserved the complete experimental design.

In [24]:
# ============================================================
# Dataset Preservation Verification
# ============================================================

preservation_summary = (

    cleaned_dataset

    .groupby("operating_hour")

    .size()

    .reset_index(name="Observations")

)

preservation_summary

,operating_hour,Observations
0,50,179360
1,100,179360
2,150,179360
3,200,179360
4,250,179360
5,300,179360
6,350,179360
7,400,179360
8,450,179360
9,500,179360


In [ ]:
### Dataset Integrity Summary

The integrity assessment summarises the preservation of the cleaned operational dataset following preprocessing.

This provides the final verification that the dataset accurately represents the original operational experiments and is suitable for exploratory data analysis.

In [25]:
# ============================================================
# Integrity Summary
# ============================================================

integrity_summary = pd.DataFrame({

    "Metric":[

        "Dataset dimensions verified",

        "Operating-hour experiments preserved",

        "Experimental observations preserved",

        "Overall dataset integrity",

        "Dataset readiness"

    ],

    "Value":[

        "Yes",

        cleaned_dataset["operating_hour"].nunique(),

        f"{cleaned_dataset.shape[0]:,}",

        "Verified",

        "Ready for Exploratory Data Analysis (Notebook 06)"

    ]

})

integrity_summary

,Metric,Value
0,Dataset dimensions verified,Yes
1,Operating-hour experiments preserved,20
2,Experimental observations preserved,"3,629,680"
3,Overall dataset integrity,Verified
4,Dataset readiness,Ready for Exploratory Data Analysis (Notebook 06)


In [ ]:
# Notebook Summary

This notebook evaluated the overall quality of the cleaned operational dataset generated in Notebook 04. The objective was not to modify the dataset further, but to assess its completeness, consistency, validity, and integrity before proceeding to detailed quality investigations and exploratory data analysis.

The following data quality assessments were successfully completed:

- Verified that the cleaned dataset was loaded correctly and matched the expected dimensions from the previous notebook.
- Assessed dataset completeness by examining overall completeness, variable completeness, and operating-hour experiment completeness.
- Confirmed that the dataset contains **100% complete data**, with no missing values across any variable or operating-hour experiment.
- Verified structural consistency by confirming variable names, data types, operating-hour identifiers, and overall dataset structure.
- Confirmed that all operational measurement variables remain stored as numerical (`float64`) variables, while the `operating_hour` identifier remains stored as an integer (`int64`).
- Evaluated the physical validity of the recorded measurements by examining the observed operating ranges of voltage, current, power, pressure, temperature, and reactant flow variables.
- Identified a very small number of negative current, voltage, and power measurements. These observations represent only a negligible proportion of the dataset and are documented for further investigation rather than being treated as errors at this stage.
- Verified the overall integrity of the cleaned dataset by confirming that all twenty operating-hour experiments remain present and that the original experimental structure has been preserved throughout the preprocessing workflow.

Overall, the data quality assessment indicates that the cleaned operational dataset is complete, structurally consistent, physically plausible, and has maintained its integrity following preprocessing.

The dataset is therefore considered suitable for the next phase of the project, where more detailed investigations of missing values, duplicate records, outliers, sensor behaviour, and time-series integrity will be performed before exploratory data analysis and predictive modelling.